# ARAPPAV Rollout Explorer

This notebook loads and visualizes self-play rollouts from `data/rollouts/`
to help with qualitative analysis and debugging.

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid")

from arappav.errors.taxonomy import ErrorType

In [ ]:
ROLLOUT_DIR = Path("../data/rollouts")

def load_rollouts(prefix: str = "perturber") -> list[dict]:
    """Load all rollout JSONL files matching a prefix."""
    rollouts = []
    for path in sorted(ROLLOUT_DIR.glob(f"{prefix}_*.jsonl")):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    rollouts.append(json.loads(line))
    return rollouts

# Load latest round's rollouts
p_rollouts = load_rollouts("perturber")
v_rollouts = load_rollouts("verifier")

print(f"Loaded {len(p_rollouts)} Perturber rollouts")
print(f"Loaded {len(v_rollouts)} Verifier rollouts")

## Format Compliance

What fraction of Perturber outputs parse correctly?

In [ ]:
if p_rollouts:
    valid = sum(1 for r in p_rollouts if r.get("format_valid"))
    total = len(p_rollouts)
    print(f"Perturber format compliance: {valid}/{total} ({valid/total:.1%})")

    # Show format violation reasons
    reasons = Counter(
        r.get("format_reason", "unknown")[:80]
        for r in p_rollouts
        if not r.get("format_valid")
    )
    if reasons:
        print("\nTop format violation reasons:")
        for reason, count in reasons.most_common(5):
            print(f"  [{count}x] {reason}")
else:
    print("No rollouts found. Run the self-play loop first.")

## Error Type Distribution

Which error types does the Perturber inject most frequently?

In [ ]:
if p_rollouts:
    type_counts = Counter()
    for r in p_rollouts:
        if r.get("format_valid") and r.get("ground_truth"):
            for err in r["ground_truth"]:
                type_counts[err.get("error_type", "unknown")] += 1

    if type_counts:
        types, counts = zip(*type_counts.most_common())
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.bar(types, counts)
        ax.set_title("Injected Error Types")
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=45)
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    str(count), ha="center", va="bottom", fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print("No error type data found.")
else:
    print("No rollouts found.")

## k Distribution

Distribution of the number of errors injected per episode.

In [ ]:
if p_rollouts:
    k_values = [r["k"] for r in p_rollouts]
    k_counts = Counter(k_values)
    ks, counts = zip(*sorted(k_counts.items()))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(ks, counts)
    ax.set_title("Distribution of k (errors per episode)")
    ax.set_xlabel("k")
    ax.set_ylabel("Count")
    ax.set_xticks(ks)
    plt.show()
else:
    print("No rollouts found.")

## Verifier Performance

How many claims does the Verifier make, and how do they relate to k?

In [ ]:
if v_rollouts:
    for r in v_rollouts[:5]:
        k = r.get("k", "?")
        n_responses = len(r.get("responses", []))
        print(f"Paper: {r.get('paper_id', '?')}, k={k}, responses sampled={n_responses}")
        for i, (raw, vout, err) in enumerate(r.get("responses", [])):
            if vout:
                print(f"  Response {i}: {len(vout.claims) if hasattr(vout, 'claims') else '?'} claims")
            else:
                print(f"  Response {i}: PARSE ERROR — {err[:100] if err else 'unknown'}")
else:
    print("No Verifier rollouts found.")

## Debug: Inspect a Single Episode

Print a full episode (original, perturbed, ground truth, verifier claims).

In [ ]:
def print_episode(rollout_idx: int = 0):
    """Print a detailed view of a single rollout episode."""
    if not p_rollouts:
        print("No rollouts available.")
        return

    r = p_rollouts[rollout_idx % len(p_rollouts)]

    print("=" * 70)
    print(f"Episode: {r.get('rollout_id', '?')}  |  Paper: {r.get('paper_id', '?')}  |  k={r.get('k', '?')}")
    print(f"Format valid: {r.get('format_valid', False)}")
    print("=" * 70)

    if r.get("original_text"):
        print("\n--- ORIGINAL TEXT (first 500 chars) ---")
        print(r["original_text"][:500])

    if r.get("perturbed_text"):
        print("\n--- PERTURBED TEXT (first 500 chars) ---")
        print(r["perturbed_text"][:500])

    if r.get("ground_truth"):
        print(f"\n--- GROUND TRUTH ({len(r['ground_truth'])} errors) ---")
        for err in r["ground_truth"]:
            print(f"  [{err.get('error_type', '?')}] {err.get('error_id', '?')}: "
                  f"'{err.get('injected_text', '')[:100]}'")
            print(f"    Rationale: {err.get('rationale', '')[:200]}")

print_episode(0)

## Summary Statistics

Aggregate stats across all loaded rollouts.

In [ ]:
if p_rollouts:
    total = len(p_rollouts)
    valid = sum(1 for r in p_rollouts if r.get("format_valid"))
    total_errors = sum(len(r.get("ground_truth", [])) for r in p_rollouts)

    print(f"Total episodes:        {total}")
    print(f"Format-valid:          {valid} ({valid/total:.1%})")
    print(f"Total errors injected: {total_errors}")
    print(f"Avg errors/episode:    {total_errors/max(1,total):.1f}")

    # Papers used
    papers = set(r.get("paper_id") for r in p_rollouts)
    print(f"Unique papers:         {len(papers)}")

    # k distribution summary
    ks = [r["k"] for r in p_rollouts]
    print(f"k range:               [{min(ks)}, {max(ks)}]")
    print(f"k mean:                {np.mean(ks):.1f}")
else:
    print("No rollouts loaded.")